
# Supplementary repeated stratified 5×5 CV stability analysis — screening operating point

This notebook runs a supplementary stability analysis for the **final screening-oriented modeling strategy**.

It does **not** replace the final 75/25 held-out internal test evaluation. Its purpose is to check whether the high-sensitivity screening operating point remains broadly stable across alternative stratified partitions.

Key safeguards:

- Repeated stratified CV: 5 folds × 5 repeats = 25 validation partitions.
- Scaling is fitted inside each training fold only.
- PCA is fitted inside each training fold only.
- The number of PCs is fixed to the final retained component counts per outcome: victimization = 18, perpetration = 22, overlap = 18.
- The operating threshold is selected **within the training fold only** using a screening objective:
  - victimization: target recall ≥ 85%
  - perpetration: target recall ≥ 90%
  - overlap: target recall ≥ 80%
- The selected fold-internal threshold is then applied to the corresponding validation fold.

This is **not nested model selection** and should not be described as external validation. It is a supplementary internal stability check.


In [1]:

# ============================================================
# CONFIG
# ============================================================
from pathlib import Path
import os
import json
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd().resolve()
OUTPUT_DIR = BASE_DIR / "final_repeated_cv_screening_operating_point_PCA_trainonly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5

# Use the final retained component counts, not a fresh 95% rule that may change the operating point.
FINAL_N_COMPONENTS = {
    "victimization": 18,
    "perpetration": 22,
    "overlap": 18,
}

# Fold-internal screening thresholds: selected only from training-fold data.
TARGET_RECALL = {
    "victimization": 0.85,
    "perpetration": 0.90,
    "overlap": 0.80,
}

THRESHOLD_GRID = np.linspace(0.0, 1.0, 1001)
INNER_TUNE_SIZE = 0.20

# Perpetration DNN settings. Reduce EPOCHS for quick smoke tests; use 80-150 for final.
RUN_PERPETRATION_DNN = True
DNN_EPOCHS = 120
DNN_BATCH_SIZE = 64
DNN_PATIENCE = 15
DNN_POS_WEIGHT_MULTIPLIER = 1.0
DNN_LEARNING_RATE = 0.001

# Victimization tree internal pruning selection.
TREE_ALPHA_MAX_CANDIDATES = 60

# Overlap logistic sample weighting.
OVERLAP_POS_WEIGHT_MULTIPLIER = 1.5

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FINAL_N_COMPONENTS:", FINAL_N_COMPONENTS)
print("TARGET_RECALL:", TARGET_RECALL)


BASE_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final
OUTPUT_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_repeated_cv_screening_operating_point_PCA_trainonly
FINAL_N_COMPONENTS: {'victimization': 18, 'perpetration': 22, 'overlap': 18}
TARGET_RECALL: {'victimization': 0.85, 'perpetration': 0.9, 'overlap': 0.8}


In [2]:

# ============================================================
# DATA LOADING AND ANALYTIC MATRIX CONSTRUCTION
# ============================================================

def find_data_dir(start: Path) -> Path:
    candidates = []
    p = start.resolve()
    for _ in range(7):
        candidates.append(p / "data")
        p = p.parent
    for c in candidates:
        if (c / "lista_global_vars.csv").exists() and (c / "target_col.csv").exists():
            return c
    raise FileNotFoundError("Could not find data/lista_global_vars.csv and data/target_col.csv")

DATA_DIR = find_data_dir(BASE_DIR)
FEATURES_PATH = DATA_DIR / "lista_global_vars.csv"
TARGET_PATH = DATA_DIR / "target_col.csv"

feat_df = pd.read_csv(FEATURES_PATH)
target_df = pd.read_csv(TARGET_PATH).fillna(0)

print("DATA_DIR:", DATA_DIR)
print("feat_df:", feat_df.shape)
print("target_df:", target_df.shape)
print("feature columns:", list(feat_df.columns))
print("target columns:", list(target_df.columns))

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"
TARGET_DROP_COLS = [
    "VÍCTIMA", "PERPETRADOR", "VICTIMA_PERPETRADOR", "POLIVICTIMIZACION",
    "POLIPERPETRACION", "SOLO.VICTIMA", "SOLO.PERPETRADOR", "NO.VICT_NO.PERP",
    "V.O", "P.SUM.TOTAL", "V.SUM.TOTAL"
]

if V_COL not in target_df.columns or P_COL not in target_df.columns:
    raise ValueError("target_col.csv must contain V.SUM.TOTAL and P.SUM.TOTAL")

# Reconstruct analytic matrix as in final notebooks: feature matrix + count-based targets.
df_merged = feat_df.join(target_df, how="inner")

# Remove very rare categories as in the final analytic matrix construction.
rare_mask = pd.Series(False, index=df_merged.index)
if "GENERO_BIN_2" in df_merged.columns:
    rare_mask = rare_mask | (pd.to_numeric(df_merged["GENERO_BIN_2"], errors="coerce") == 1)
if "ORIENTSEX.BN_3" in df_merged.columns:
    rare_mask = rare_mask | (pd.to_numeric(df_merged["ORIENTSEX.BN_3"], errors="coerce") == 1)

print("Rows removed by rare category filter:", int(rare_mask.sum()))
df_merged = df_merged.loc[~rare_mask].copy().reset_index(drop=True)

for c in ["GENERO_BIN_2", "ORIENTSEX.BN_3"]:
    if c in df_merged.columns:
        df_merged = df_merged.drop(columns=[c])

df_merged[V_COL] = pd.to_numeric(df_merged[V_COL], errors="coerce").fillna(0)
df_merged[P_COL] = pd.to_numeric(df_merged[P_COL], errors="coerce").fillna(0)

y_victim = (df_merged[V_COL] >= 1).astype(int)
y_perp = (df_merged[P_COL] >= 1).astype(int)
y_overlap = ((df_merged[V_COL] >= 1) & (df_merged[P_COL] >= 1)).astype(int)

df_features = df_merged.drop(columns=[c for c in TARGET_DROP_COLS if c in df_merged.columns], errors="ignore").copy()
for c in ["INTERSECT", "victim_count_ge1", "perp_count_ge1", "overlap_count_ge1"]:
    if c in df_features.columns:
        df_features = df_features.drop(columns=[c])

# Reproduce final feature recoding used in the PCA notebooks.
for c, mapping in {
    "PAÍS": {1: True, 2: False},
    "ETNIA.BN": {0.0: False, 1.0: True},
    "FUGAS.BN": {0.0: False, 1.0: True},
}.items():
    if c in df_features.columns:
        df_features[c] = df_features[c].replace(mapping)

if "GENERO_BIN_0" in df_features.columns:
    df_features["GENERO.BN0"] = df_features["GENERO_BIN_0"].replace({0.0: False, 1.0: True})
if "GENERO_BIN_1" in df_features.columns:
    df_features["GENERO.BN1"] = df_features["GENERO_BIN_1"].replace({0.0: False, 1.0: True})
if "ORIENTSEX.BN_1" in df_features.columns:
    df_features["ORIENTSEX.BN0"] = df_features["ORIENTSEX.BN_1"].replace({0.0: False, 1.0: True})
if "ORIENTSEX.BN_2" in df_features.columns:
    df_features["ORIENTSEX.BN1"] = df_features["ORIENTSEX.BN_2"].replace({0.0: False, 1.0: True})

for c in ["GENERO_BIN_0", "GENERO_BIN_1", "ORIENTSEX.BN_1", "ORIENTSEX.BN_2"]:
    if c in df_features.columns:
        df_features = df_features.drop(columns=[c])

if "CONVIVEN.5" in df_features.columns:
    df_features = df_features.rename(columns={"CONVIVEN.5": "CONVIVEN_H"})
    df_features["CONVIVEN_H"] = df_features["CONVIVEN_H"].replace({0.0: False, 1.0: True})
if "CONVIVEN.6" in df_features.columns:
    df_features = df_features.rename(columns={"CONVIVEN.6": "CONVIVEN_0"})
    df_features["CONVIVEN_0"] = df_features["CONVIVEN_0"].replace({0.0: False, 1.0: True})

for c in df_features.columns:
    if df_features[c].dtype == "bool":
        df_features[c] = df_features[c].astype(int)
    else:
        df_features[c] = pd.to_numeric(df_features[c], errors="coerce")

missing_before = int(df_features.isna().sum().sum())
if missing_before > 0:
    print("WARNING: missing predictor values filled with 0:", missing_before)
    df_features = df_features.fillna(0)

X = df_features.astype(float).reset_index(drop=True)
outcomes = {
    "victimization": y_victim.astype(int).reset_index(drop=True),
    "perpetration": y_perp.astype(int).reset_index(drop=True),
    "overlap": y_overlap.astype(int).reset_index(drop=True),
}

print("Analytical feature matrix:", X.shape)
print("Predictor columns:", list(X.columns))
for name, y in outcomes.items():
    print(name, "positives:", int(y.sum()), "/", len(y), "prevalence:", round(float(y.mean()), 4))

X.to_csv(OUTPUT_DIR / "stability_analysis_feature_matrix.csv", index=False)
pd.DataFrame({k: v for k, v in outcomes.items()}).to_csv(OUTPUT_DIR / "stability_analysis_targets.csv", index=False)
with open(OUTPUT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(list(X.columns), f, ensure_ascii=False, indent=2)


DATA_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data
feat_df: (4024, 29)
target_df: (4024, 11)
feature columns: ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', 'CONVIVEN.5', 'CONVIVEN.6', 'AUTOEFIC.MEAN', 'AUTOEFIC.VAR', 'IMPULS.MEAN', 'IMPULS.MEDIAN', 'IMPULS.VAR', 'APOYO.MEAN', 'APOYO.MEDIAN', 'APOYO.VAR', 'MORAL.MEAN', 'MORAL.VAR', 'PORNO.T', 'GENERO_BIN_0', 'GENERO_BIN_1', 'GENERO_BIN_2', 'ORIENTSEX.BN_1', 'ORIENTSEX.BN_2', 'ORIENTSEX.BN_3']
target columns: ['VÍCTIMA', 'PERPETRADOR', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION', 'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O', 'P.SUM.TOTAL', 'V.SUM.TOTAL']
Rows removed by rare category filter: 257
Analytical feature matrix: (3767, 27)
Predictor columns: ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', '

In [3]:

# ============================================================
# METRICS, PCA, THRESHOLD SELECTION, AND MODEL HELPERS
# ============================================================

def fit_fold_pca_fixed_components(X_train_df, X_test_df, n_keep):
    """Fit scaler + PCA on training fold only, then keep the final-model component count."""
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_df.values)
    X_test_scaled = scaler.transform(X_test_df.values)

    means = X_train_scaled.mean(axis=0)
    X_train_centered = X_train_scaled - means
    X_test_centered = X_test_scaled - means

    n_all = X_train_df.shape[1]
    pca = PCA(n_components=n_all, random_state=RANDOM_STATE)
    X_train_pca_all = pca.fit_transform(X_train_centered)
    X_test_pca_all = pca.transform(X_test_centered)

    n_keep = int(min(n_keep, X_train_pca_all.shape[1]))
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    retained_var = float(cum_var[n_keep - 1])

    return X_train_pca_all[:, :n_keep], X_test_pca_all[:, :n_keep], n_keep, retained_var


def binary_metrics_from_prob(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    return {
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "recall_sensitivity": float(recall),
        "specificity": float(specificity),
        "precision_ppv": float(ppv) if pd.notna(ppv) else np.nan,
        "npv": float(npv) if pd.notna(npv) else np.nan,
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_positive": float(f1) if pd.notna(f1) else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc) if pd.notna(roc_auc) else np.nan,
        "pr_auc_average_precision": float(pr_auc) if pd.notna(pr_auc) else np.nan,
        "positive_support": int(np.sum(y_true == 1)),
        "negative_support": int(np.sum(y_true == 0)),
        "support": int(len(y_true)),
    }


def select_screening_threshold(y_true, y_prob, target_recall, threshold_grid=THRESHOLD_GRID):
    """Select threshold using training-fold/tuning-fold data only.

    Rule:
    - Prefer thresholds with recall >= target_recall and specificity > 0.
    - Among eligible thresholds, maximize balanced accuracy, then specificity, then PPV.
    - If no threshold reaches the target with non-zero specificity, choose the threshold with
      highest recall among thresholds with specificity > 0, then balanced accuracy.
    """
    rows = []
    for thr in threshold_grid:
        m = binary_metrics_from_prob(y_true, y_prob, threshold=float(thr))
        rows.append({"threshold": float(thr), **m})

    tab = pd.DataFrame(rows)
    eligible = tab[(tab["recall_sensitivity"] >= target_recall) & (tab["specificity"] > 0)]

    if len(eligible) > 0:
        best = eligible.sort_values(
            ["balanced_accuracy", "specificity", "precision_ppv", "threshold"],
            ascending=[False, False, False, False]
        ).iloc[0]
        status = "target_met"
    else:
        nonzero = tab[tab["specificity"] > 0]
        if len(nonzero) > 0:
            best = nonzero.sort_values(
                ["recall_sensitivity", "balanced_accuracy", "specificity"],
                ascending=[False, False, False]
            ).iloc[0]
            status = "target_not_met_nonzero_specificity"
        else:
            best = tab.sort_values(["recall_sensitivity", "balanced_accuracy"], ascending=[False, False]).iloc[0]
            status = "target_not_met_zero_specificity"

    return float(best["threshold"]), best.to_dict(), tab, status


def inner_tune_split_indices(y_train, seed):
    y_train = np.asarray(y_train).astype(int)
    return train_test_split(
        np.arange(len(y_train)),
        test_size=INNER_TUNE_SIZE,
        random_state=seed,
        stratify=y_train,
    )


def choose_tree_alpha_and_threshold(X_train_pca, y_train, target_recall, seed):
    y_train = np.asarray(y_train).astype(int)
    fit_idx, tune_idx = inner_tune_split_indices(y_train, seed)
    X_fit, y_fit = X_train_pca[fit_idx], y_train[fit_idx]
    X_tune, y_tune = X_train_pca[tune_idx], y_train[tune_idx]

    base_tree = DecisionTreeClassifier(random_state=seed, class_weight=None)
    path = base_tree.cost_complexity_pruning_path(X_fit, y_fit)
    alphas = np.unique(path.ccp_alphas)
    alphas = alphas[np.isfinite(alphas)]
    alphas = alphas[alphas >= 0]

    if len(alphas) > TREE_ALPHA_MAX_CANDIDATES:
        alphas = np.unique(np.quantile(alphas, np.linspace(0, 1, TREE_ALPHA_MAX_CANDIDATES)))

    rows = []
    threshold_tables = []

    for alpha in alphas:
        clf = DecisionTreeClassifier(random_state=seed, ccp_alpha=float(alpha))
        clf.fit(X_fit, y_fit)
        y_prob_tune = clf.predict_proba(X_tune)[:, 1]
        thr, thr_metrics, thr_tab, status = select_screening_threshold(y_tune, y_prob_tune, target_recall)

        row = {
            "ccp_alpha": float(alpha),
            "selected_threshold": float(thr),
            "threshold_status": status,
            **{f"tune_{k}": v for k, v in thr_metrics.items() if k != "threshold"},
        }
        rows.append(row)
        tmp = thr_tab.copy()
        tmp["ccp_alpha"] = float(alpha)
        threshold_tables.append(tmp)

    cand = pd.DataFrame(rows)
    if cand.empty:
        return 0.0, 0.5, cand, pd.DataFrame()

    eligible = cand[(cand["threshold_status"] == "target_met")]
    if len(eligible) > 0:
        best = eligible.sort_values(
            ["tune_balanced_accuracy", "tune_specificity", "tune_precision_ppv"],
            ascending=[False, False, False]
        ).iloc[0]
    else:
        best = cand.sort_values(
            ["tune_recall_sensitivity", "tune_balanced_accuracy", "tune_specificity"],
            ascending=[False, False, False]
        ).iloc[0]

    all_thresholds = pd.concat(threshold_tables, ignore_index=True) if threshold_tables else pd.DataFrame()
    return float(best["ccp_alpha"]), float(best["selected_threshold"]), cand, all_thresholds


def fit_predict_victim_tree_screening(X_train_pca, X_test_pca, y_train, seed):
    target_recall = TARGET_RECALL["victimization"]
    alpha, thr, alpha_table, threshold_table = choose_tree_alpha_and_threshold(
        X_train_pca, y_train, target_recall, seed
    )
    clf = DecisionTreeClassifier(random_state=seed, ccp_alpha=alpha)
    clf.fit(X_train_pca, y_train)
    y_prob = clf.predict_proba(X_test_pca)[:, 1]
    return y_prob, {
        "selected_threshold": float(thr),
        "target_recall": float(target_recall),
        "selected_ccp_alpha": float(alpha),
        "inner_alpha_table": alpha_table,
        "inner_threshold_table": threshold_table,
    }


def fit_weighted_logreg(X_train_pca, y_train, pos_weight_multiplier, seed):
    y_train = np.asarray(y_train).astype(int)
    pos_rate = y_train.mean()
    base_pos_weight = (1.0 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
    sample_weight = np.where(y_train == 1, base_pos_weight * pos_weight_multiplier, 1.0)
    model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=seed)
    model.fit(X_train_pca, y_train, sample_weight=sample_weight)
    return model, float(base_pos_weight * pos_weight_multiplier)


def fit_predict_overlap_logreg_screening(X_train_pca, X_test_pca, y_train, seed):
    target_recall = TARGET_RECALL["overlap"]
    y_train = np.asarray(y_train).astype(int)

    fit_idx, tune_idx = inner_tune_split_indices(y_train, seed)
    X_fit, y_fit = X_train_pca[fit_idx], y_train[fit_idx]
    X_tune, y_tune = X_train_pca[tune_idx], y_train[tune_idx]

    tune_model, tune_pos_weight = fit_weighted_logreg(X_fit, y_fit, OVERLAP_POS_WEIGHT_MULTIPLIER, seed)
    y_prob_tune = tune_model.predict_proba(X_tune)[:, 1]
    thr, thr_metrics, thr_tab, status = select_screening_threshold(y_tune, y_prob_tune, target_recall)

    final_model, final_pos_weight = fit_weighted_logreg(X_train_pca, y_train, OVERLAP_POS_WEIGHT_MULTIPLIER, seed)
    y_prob = final_model.predict_proba(X_test_pca)[:, 1]

    return y_prob, {
        "selected_threshold": float(thr),
        "target_recall": float(target_recall),
        "threshold_status": status,
        "positive_class_weight": float(final_pos_weight),
        "tune_positive_class_weight": float(tune_pos_weight),
        "inner_threshold_table": thr_tab,
        **{f"tune_{k}": v for k, v in thr_metrics.items() if k != "threshold"},
    }


def build_dnn(input_dim, seed):
    import tensorflow as tf
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dropout(0.15),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=DNN_LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.Recall(name="recall")],
    )
    return model


def train_dnn_model(X_train_pca, y_train, seed):
    import tensorflow as tf
    y_train = np.asarray(y_train).astype(int)
    pos_rate = y_train.mean()
    base_pos_weight = (1 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
    class_weight = {0: 1.0, 1: float(base_pos_weight * DNN_POS_WEIGHT_MULTIPLIER)}

    model = build_dnn(X_train_pca.shape[1], seed)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_recall",
            mode="max",
            patience=DNN_PATIENCE,
            restore_best_weights=True,
            verbose=0,
        )
    ]
    hist = model.fit(
        X_train_pca,
        y_train,
        epochs=DNN_EPOCHS,
        batch_size=DNN_BATCH_SIZE,
        validation_split=0.20,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0,
    )
    return model, class_weight, int(len(hist.history.get("loss", [])))


def fit_predict_perp_dnn_screening(X_train_pca, X_test_pca, y_train, seed):
    target_recall = TARGET_RECALL["perpetration"]
    y_train = np.asarray(y_train).astype(int)

    if not RUN_PERPETRATION_DNN:
        fit_idx, tune_idx = inner_tune_split_indices(y_train, seed)
        X_fit, y_fit = X_train_pca[fit_idx], y_train[fit_idx]
        X_tune, y_tune = X_train_pca[tune_idx], y_train[tune_idx]

        tune_model, tune_pos_weight = fit_weighted_logreg(X_fit, y_fit, 1.0, seed)
        y_prob_tune = tune_model.predict_proba(X_tune)[:, 1]
        thr, thr_metrics, thr_tab, status = select_screening_threshold(y_tune, y_prob_tune, target_recall)
        final_model, final_pos_weight = fit_weighted_logreg(X_train_pca, y_train, 1.0, seed)
        y_prob = final_model.predict_proba(X_test_pca)[:, 1]
        return y_prob, {
            "selected_threshold": float(thr),
            "target_recall": float(target_recall),
            "threshold_status": status,
            "fallback": "weighted_logistic_regression",
            "positive_class_weight": float(final_pos_weight),
            "inner_threshold_table": thr_tab,
            **{f"tune_{k}": v for k, v in thr_metrics.items() if k != "threshold"},
        }

    fit_idx, tune_idx = inner_tune_split_indices(y_train, seed)
    X_fit, y_fit = X_train_pca[fit_idx], y_train[fit_idx]
    X_tune, y_tune = X_train_pca[tune_idx], y_train[tune_idx]

    tune_model, tune_cw, tune_epochs = train_dnn_model(X_fit, y_fit, seed)
    y_prob_tune = tune_model.predict(X_tune, verbose=0).reshape(-1)
    thr, thr_metrics, thr_tab, status = select_screening_threshold(y_tune, y_prob_tune, target_recall)

    final_model, final_cw, final_epochs = train_dnn_model(X_train_pca, y_train, seed + 10000)
    y_prob = final_model.predict(X_test_pca, verbose=0).reshape(-1)

    return y_prob, {
        "selected_threshold": float(thr),
        "target_recall": float(target_recall),
        "threshold_status": status,
        "class_weight_positive": float(final_cw[1]),
        "tune_class_weight_positive": float(tune_cw[1]),
        "epochs_ran": int(final_epochs),
        "tune_epochs_ran": int(tune_epochs),
        "inner_threshold_table": thr_tab,
        **{f"tune_{k}": v for k, v in thr_metrics.items() if k != "threshold"},
    }


In [4]:

# ============================================================
# RUN REPEATED STRATIFIED 5×5 CV — SCREENING OPERATING POINT
# ============================================================

rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

fold_rows = []
alpha_rows = []
threshold_rows = []

for outcome_name, y_series in outcomes.items():
    y = y_series.to_numpy().astype(int)
    print("\n" + "=" * 90)
    print("Outcome:", outcome_name, "n=", len(y), "positives=", int(y.sum()), "target_recall=", TARGET_RECALL[outcome_name])

    for fold_id, (train_idx, test_idx) in enumerate(rskf.split(X, y), start=1):
        seed = RANDOM_STATE + 1000 * list(outcomes.keys()).index(outcome_name) + fold_id

        X_train_df = X.iloc[train_idx].copy()
        X_test_df = X.iloc[test_idx].copy()
        y_train = y[train_idx]
        y_test = y[test_idx]

        n_keep_target = FINAL_N_COMPONENTS[outcome_name]
        X_train_pca, X_test_pca, n_components, retained_var = fit_fold_pca_fixed_components(
            X_train_df,
            X_test_df,
            n_keep=n_keep_target,
        )

        if outcome_name == "victimization":
            y_prob, info = fit_predict_victim_tree_screening(X_train_pca, X_test_pca, y_train, seed)
            model_label = "pruned_decision_tree_screening_threshold_fold_internal"

            if isinstance(info.get("inner_alpha_table"), pd.DataFrame):
                tmp = info["inner_alpha_table"].copy()
                tmp.insert(0, "outcome", outcome_name)
                tmp.insert(1, "fold_id", fold_id)
                alpha_rows.append(tmp)

        elif outcome_name == "perpetration":
            y_prob, info = fit_predict_perp_dnn_screening(X_train_pca, X_test_pca, y_train, seed)
            model_label = "compact_dnn_screening_threshold_fold_internal" if RUN_PERPETRATION_DNN else "weighted_logreg_fallback_screening_threshold"

        elif outcome_name == "overlap":
            y_prob, info = fit_predict_overlap_logreg_screening(X_train_pca, X_test_pca, y_train, seed)
            model_label = "weighted_logistic_regression_SW_pos1.5_screening_threshold_fold_internal"

        else:
            raise ValueError(outcome_name)

        selected_threshold = float(info.get("selected_threshold", 0.5))
        m = binary_metrics_from_prob(y_test, y_prob, threshold=selected_threshold)

        row = {
            "outcome": outcome_name,
            "fold_id": fold_id,
            "model_label": model_label,
            "selected_threshold": selected_threshold,
            "target_recall": float(TARGET_RECALL[outcome_name]),
            "threshold_status": info.get("threshold_status", "not_applicable"),
            "n_components": int(n_components),
            "target_n_components": int(n_keep_target),
            "retained_variance": float(retained_var),
            "train_n": int(len(train_idx)),
            "test_n": int(len(test_idx)),
            "train_positive_prevalence": float(np.mean(y_train)),
            "test_positive_prevalence": float(np.mean(y_test)),
            **m,
        }

        for k, v in info.items():
            if isinstance(v, (int, float, str, np.integer, np.floating)):
                row[k] = v

        fold_rows.append(row)

        if isinstance(info.get("inner_threshold_table"), pd.DataFrame):
            tmp_thr = info["inner_threshold_table"].copy()
            tmp_thr.insert(0, "outcome", outcome_name)
            tmp_thr.insert(1, "fold_id", fold_id)
            tmp_thr["selected_threshold"] = selected_threshold
            threshold_rows.append(tmp_thr)

        print(
            f"{outcome_name:14s} fold {fold_id:02d} | "
            f"thr={selected_threshold:.3f} | "
            f"recall={m['recall_sensitivity']:.3f} spec={m['specificity']:.3f} "
            f"ppv={m['precision_ppv']:.3f} ba={m['balanced_accuracy']:.3f} "
            f"auc={m['roc_auc']:.3f} ap={m['pr_auc_average_precision']:.3f}"
        )

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(OUTPUT_DIR / "repeated_5x5_cv_screening_fold_metrics.csv", index=False)

if alpha_rows:
    alpha_all = pd.concat(alpha_rows, ignore_index=True)
    alpha_all.to_csv(OUTPUT_DIR / "victimization_tree_inner_alpha_selection_tables.csv", index=False)
else:
    alpha_all = pd.DataFrame()

if threshold_rows:
    threshold_all = pd.concat(threshold_rows, ignore_index=True)
    threshold_all.to_csv(OUTPUT_DIR / "inner_threshold_selection_tables_all_outcomes.csv", index=False)
else:
    threshold_all = pd.DataFrame()

print("\nSaved fold metrics to:")
print(OUTPUT_DIR / "repeated_5x5_cv_screening_fold_metrics.csv")
print("Fold metrics shape:", fold_metrics.shape)



Outcome: victimization n= 3767 positives= 1861 target_recall= 0.85
victimization  fold 01 | thr=0.336 | recall=0.815 spec=0.403 ppv=0.571 ba=0.609 auc=0.694 ap=0.661
victimization  fold 02 | thr=0.329 | recall=0.890 spec=0.291 ppv=0.551 ba=0.591 auc=0.710 ap=0.685
victimization  fold 03 | thr=0.304 | recall=0.903 spec=0.239 ppv=0.537 ba=0.571 auc=0.702 ap=0.676
victimization  fold 04 | thr=0.254 | recall=0.906 spec=0.273 ppv=0.549 ba=0.589 auc=0.701 ap=0.663
victimization  fold 05 | thr=0.283 | recall=0.876 spec=0.325 ppv=0.559 ba=0.601 auc=0.658 ap=0.644
victimization  fold 06 | thr=0.235 | recall=0.917 spec=0.202 ppv=0.528 ba=0.559 auc=0.691 ap=0.673
victimization  fold 07 | thr=0.309 | recall=0.831 spec=0.323 ppv=0.546 ba=0.577 auc=0.680 ap=0.656
victimization  fold 08 | thr=0.257 | recall=0.898 spec=0.249 ppv=0.539 ba=0.574 auc=0.654 ap=0.616
victimization  fold 09 | thr=0.223 | recall=0.909 spec=0.318 ppv=0.565 ba=0.613 auc=0.707 ap=0.696
victimization  fold 10 | thr=0.287 | reca

E0000 00:00:1783669842.829185 13900720 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


perpetration   fold 01 | thr=0.489 | recall=0.678 spec=0.672 ppv=0.388 ba=0.675 auc=0.720 ap=0.447


E0000 00:00:1783669849.231705 13900720 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


perpetration   fold 02 | thr=0.498 | recall=0.616 spec=0.671 ppv=0.365 ba=0.643 auc=0.708 ap=0.417


...

perpetration   fold 23 | thr=0.291 | recall=0.853 spec=0.385 ppv=0.299 ba=0.619 auc=0.704 ap=0.410


E0000 00:00:1783669943.196730 13900720 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


perpetration   fold 24 | thr=0.363 | recall=0.938 spec=0.217 ppv=0.269 ba=0.577 auc=0.701 ap=0.418


E0000 00:00:1783669948.561989 13900720 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


perpetration   fold 25 | thr=0.346 | recall=1.000 spec=0.000 ppv=0.235 ba=0.500 auc=0.625 ap=0.332

Outcome: overlap n= 3767 positives= 713 target_recall= 0.8
overlap        fold 01 | thr=0.486 | recall=0.790 spec=0.502 ppv=0.271 ba=0.646 auc=0.743 ap=0.413
overlap        fold 02 | thr=0.473 | recall=0.797 spec=0.494 ppv=0.270 ba=0.646 auc=0.746 ap=0.403
overlap        fold 03 | thr=0.511 | recall=0.817 spec=0.576 ppv=0.309 ba=0.697 auc=0.759 ap=0.432
overlap        fold 04 | thr=0.456 | recall=0.810 spec=0.476 ppv=0.264 ba=0.643 auc=0.742 ap=0.438
overlap        fold 05 | thr=0.510 | recall=0.790 spec=0.559 ppv=0.296 ba=0.675 auc=0.744 ap=0.416
overlap        fold 06 | thr=0.446 | recall=0.832 spec=0.455 ppv=0.263 ba=0.644 auc=0.736 ap=0.396
overlap        fold 07 | thr=0.509 | recall=0.818 spec=0.555 ppv=0.301 ba=0.687 auc=0.761 ap=0.443
overlap        fold 08 | thr=0.414 | recall=0.852 spec=0.376 ppv=0.241 ba=0.614 auc=0.714 ap=0.363
overlap        fold 09 | thr=0.499 | recall=0.873

In [6]:
from datetime import datetime
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

2026-07-10 09:53:11


In [5]:

# ============================================================
# SUMMARIZE CV STABILITY RESULTS
# ============================================================

metric_cols = [
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "roc_auc",
    "pr_auc_average_precision",
    "selected_threshold",
    "n_components",
    "retained_variance",
]

summary_rows = []
for outcome, sub in fold_metrics.groupby("outcome"):
    for metric in metric_cols:
        vals = pd.to_numeric(sub[metric], errors="coerce").dropna().to_numpy()
        summary_rows.append({
            "outcome": outcome,
            "metric": metric,
            "n_partitions": int(len(vals)),
            "mean": float(np.mean(vals)),
            "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "median": float(np.median(vals)),
            "p2_5": float(np.percentile(vals, 2.5)),
            "p97_5": float(np.percentile(vals, 97.5)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "repeated_5x5_cv_screening_summary_long.csv", index=False)

pretty_metrics = [
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "roc_auc",
    "pr_auc_average_precision",
]

pretty = summary[summary["metric"].isin(pretty_metrics)].copy()
pretty["formatted_mean_sd"] = pretty.apply(
    lambda r: f"{r['mean']*100:.1f}% ({r['sd']*100:.1f})" if pd.notna(r["sd"]) else f"{r['mean']*100:.1f}%",
    axis=1,
)
pretty["formatted_p2_5_p97_5"] = pretty.apply(
    lambda r: f"{r['mean']*100:.1f}% ({r['p2_5']*100:.1f}–{r['p97_5']*100:.1f})",
    axis=1,
)

pretty_wide_mean_sd = pretty.pivot(index="outcome", columns="metric", values="formatted_mean_sd").reset_index()
pretty_wide_interval = pretty.pivot(index="outcome", columns="metric", values="formatted_p2_5_p97_5").reset_index()

ordered_cols = [
    "outcome",
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "roc_auc",
    "pr_auc_average_precision",
]

pretty_wide_mean_sd = pretty_wide_mean_sd[ordered_cols]
pretty_wide_interval = pretty_wide_interval[ordered_cols]

pretty_wide_mean_sd.to_csv(OUTPUT_DIR / "repeated_5x5_cv_screening_pretty_mean_sd.csv", index=False)
pretty_wide_interval.to_csv(OUTPUT_DIR / "repeated_5x5_cv_screening_pretty_mean_p2_5_p97_5.csv", index=False)

pca_threshold_summary = summary[summary["metric"].isin(["selected_threshold", "n_components", "retained_variance"])].copy()
pca_threshold_summary.to_csv(OUTPUT_DIR / "repeated_5x5_cv_screening_threshold_pca_summary.csv", index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2400)

print("\n=== REPEATED 5×5 CV SCREENING SUMMARY — MEAN (SD) ===")
print(pretty_wide_mean_sd.to_string(index=False))

print("\n=== REPEATED 5×5 CV SCREENING SUMMARY — MEAN (2.5th–97.5th percentile) ===")
print(pretty_wide_interval.to_string(index=False))

print("\n=== THRESHOLD / PCA RETENTION SUMMARY ===")
print(pca_threshold_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Compare CV means with final held-out point estimates for quick sanity check.
final_reference = pd.DataFrame([
    {"outcome": "victimization", "final_recall": 0.871, "final_specificity": 0.310, "final_balanced_accuracy": 0.591},
    {"outcome": "perpetration", "final_recall": 0.914, "final_specificity": 0.301, "final_balanced_accuracy": 0.607},
    {"outcome": "overlap", "final_recall": 0.809, "final_specificity": 0.535, "final_balanced_accuracy": 0.672},
])

cv_core = (
    fold_metrics
    .groupby("outcome", as_index=False)
    .agg(
        cv_recall_mean=("recall_sensitivity", "mean"),
        cv_specificity_mean=("specificity", "mean"),
        cv_balanced_accuracy_mean=("balanced_accuracy", "mean"),
        cv_threshold_mean=("selected_threshold", "mean"),
    )
)
comparison = final_reference.merge(cv_core, on="outcome", how="left")
comparison.to_csv(OUTPUT_DIR / "final_heldout_vs_repeated_cv_screening_comparison.csv", index=False)

print("\n=== FINAL HELD-OUT VS REPEATED CV SCREENING COMPARISON ===")
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

readme = f"""
Supplementary repeated stratified 5×5 CV screening operating-point stability analysis

This analysis uses 5 folds repeated 5 times. Within each outer training fold, scaling and PCA are fitted only on the training fold.
The number of retained PCA components is fixed to the final model counts: victimization=18, perpetration=22, overlap=18.
For each fold, the screening threshold is selected using training-fold data only, with target recalls:
- victimization >= 0.85
- perpetration >= 0.90
- overlap >= 0.80

The selected threshold is then applied to the corresponding validation fold.
This is a supplementary stability check and does not replace the final held-out internal test evaluation.
It is not external validation and should not be described as nested model selection.
"""
with open(OUTPUT_DIR / "README_repeated_cv_screening_stability.txt", "w", encoding="utf-8") as f:
    f.write(readme)

print("\nSaved outputs to:", OUTPUT_DIR)



=== REPEATED 5×5 CV SCREENING SUMMARY — MEAN (SD) ===
      outcome recall_sensitivity  specificity precision_ppv         npv balanced_accuracy f1_positive     accuracy     roc_auc pr_auc_average_precision
      overlap        81.0% (5.0)  51.5% (7.1)   28.3% (2.1) 92.2% (1.4)       66.3% (2.3) 41.8% (2.1)  57.1% (5.1) 74.8% (1.7)              42.2% (3.0)
 perpetration       83.0% (13.4) 34.4% (26.0)   29.6% (5.3) 86.5% (3.0)       58.7% (6.5) 42.8% (3.8) 45.8% (16.8) 66.6% (5.7)              39.1% (5.2)
victimization        85.1% (7.2) 33.6% (11.3)   55.9% (2.6) 70.5% (4.7)       59.4% (2.8) 67.2% (1.7)  59.1% (2.9) 68.3% (2.1)              65.7% (2.6)

=== REPEATED 5×5 CV SCREENING SUMMARY — MEAN (2.5th–97.5th percentile) ===
      outcome recall_sensitivity       specificity     precision_ppv               npv balanced_accuracy       f1_positive          accuracy           roc_auc pr_auc_average_precision
      overlap  81.0% (72.5–89.6) 51.5% (37.6–62.7) 28.3% (24.9–31.7) 92.2% (9

In [7]:
import pandas as pd
import numpy as np

path = "/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_repeated_cv_screening_operating_point_PCA_trainonly/repeated_5x5_cv_screening_fold_metrics.csv"

df = pd.read_csv(path)

metrics = [
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
    "roc_auc",
    "pr_auc_average_precision",
]

summary_rows = []

for outcome, g in df.groupby("outcome"):
    row = {"outcome": outcome, "n_folds": len(g)}
    for m in metrics:
        if m in g.columns:
            row[f"{m}_mean"] = g[m].mean()
            row[f"{m}_sd"] = g[m].std()
            row[f"{m}_p2_5"] = np.percentile(g[m], 2.5)
            row[f"{m}_p97_5"] = np.percentile(g[m], 97.5)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

pretty = summary.copy()
for col in pretty.columns:
    if col not in ["outcome", "n_folds"]:
        pretty[col] = (pretty[col] * 100).round(1)

display(pretty)

out = path.replace("repeated_5x5_cv_screening_fold_metrics.csv", "repeated_5x5_cv_screening_summary_pretty.csv")
pretty.to_csv(out, index=False)
print("Saved:", out)

,outcome,n_folds,recall_sensitivity_mean,recall_sensitivity_sd,recall_sensitivity_p2_5,recall_sensitivity_p97_5,specificity_mean,specificity_sd,specificity_p2_5,specificity_p97_5,precision_ppv_mean,precision_ppv_sd,precision_ppv_p2_5,precision_ppv_p97_5,npv_mean,npv_sd,npv_p2_5,npv_p97_5,balanced_accuracy_mean,balanced_accuracy_sd,balanced_accuracy_p2_5,balanced_accuracy_p97_5,f1_positive_mean,f1_positive_sd,f1_positive_p2_5,f1_positive_p97_5,accuracy_mean,accuracy_sd,accuracy_p2_5,accuracy_p97_5,roc_auc_mean,roc_auc_sd,roc_auc_p2_5,roc_auc_p97_5,pr_auc_average_precision_mean,pr_auc_average_precision_sd,pr_auc_average_precision_p2_5,pr_auc_average_precision_p97_5
0,overlap,25,81.0,5.0,72.5,89.6,51.5,7.1,37.6,62.7,28.3,2.1,24.9,31.7,92.2,1.4,90.4,95.2,66.3,2.3,62.5,70.0,41.8,2.1,38.5,45.0,57.1,5.1,47.5,64.9,74.8,1.7,71.6,77.9,42.2,3.0,36.3,47.7
1,perpetration,25,83.0,13.4,62.6,100.0,34.4,26.0,0.0,67.1,29.6,5.3,23.5,38.4,86.5,3.0,NaN,NaN,58.7,6.5,50.0,67.3,42.8,3.8,37.9,49.1,45.8,16.8,23.5,66.8,66.6,5.7,54.3,72.5,39.1,5.2,27.0,45.3
2,victimization,25,85.1,7.2,69.8,92.6,33.6,11.3,16.2,54.6,55.9,2.6,51.9,61.2,70.5,4.7,61.8,77.2,59.4,2.8,54.3,64.2,67.2,1.7,63.7,70.1,59.1,2.9,53.8,64.0,68.3,2.1,64.3,71.1,65.7,2.6,60.3,69.5


Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_repeated_cv_screening_operating_point_PCA_trainonly/repeated_5x5_cv_screening_summary_pretty.csv



## Suggested manuscript language

Use this only after verifying that the output metrics are coherent and broadly consistent with the final screening operating points.

**Methods/Supplement:**

> As a supplementary stability check, we conducted repeated stratified cross-validation using 5 folds repeated 5 times. In each partition, scaling and PCA were fitted within the training fold only and then applied to the validation fold. To match the screening-oriented objective of the final models, operating thresholds were selected using training-fold data only and then applied to the corresponding validation fold. Target sensitivities were set at ≥85% for victimization, ≥90% for perpetration, and ≥80% for victim–perpetrator overlap. This analysis was used to assess stability across alternative partitions and did not replace the final held-out internal test evaluation or alter final model-selection decisions.

**Results/Supplement:**

> Repeated cross-validation was used to assess whether the screening-oriented performance pattern was stable across 25 alternative validation folds. Metrics are reported as mean, standard deviation, and empirical 2.5th–97.5th percentile intervals.

**Caution:**

> This repeated CV is an internal stability analysis. It is not external validation and should not be described as prospective or independent validation.
